# Week 2 — Hypothesis Testing, Resampling and Multiple Discovery

**Contact:** 3 hours lecture  
**Course:** MSc Astrostatistics for the Survey Era

## Learning notebooks from the Crete school
- `../../02_Hypothesis_Testing/Hypothesis.ipynb`

These Crete notebooks are **learning material**. This lecture notebook supplies the semester narrative, deeper astronomical interpretation, and the binary-SMBH case study.

## Three-hour pacing
- **0:00–0:20** — recap, motivating scientific question, diagnostic poll
- **0:20–1:05** — theory block I + worked derivation
- **1:05–1:15** — short break / discussion
- **1:15–2:00** — theory block II + computational demonstration
- **2:00–2:10** — short break
- **2:10–2:45** — binary-SMBH case study / failure mode
- **2:45–3:00** — synthesis, exit questions, bridge to the 4-hour practical

## Learning objectives

- State a null hypothesis and choose a test statistic that answers the scientific question.
- Interpret p-values correctly and distinguish statistical significance from effect size.
- Use bootstrap and permutation procedures when analytic assumptions are weak.
- Control false discoveries when many comparisons are performed.
- Design tests for SMBH mechanism sub-populations without confusing simulator labels with observational evidence.

## 1. What a hypothesis test actually answers


A classical hypothesis test begins with a model $H_0$ and a statistic $T(D)$. The p-value is

$$
p=P(T(D^\ast)\ge T(D_{\rm obs})\mid H_0),
$$

for an appropriate definition of "at least as extreme." It is **not** $P(H_0\mid D)$, and it is not the probability that the result occurred "by chance."

The first design choice is therefore not the test name but the scientific null. Examples include: the two mechanism groups have the same mass-ratio distribution; the mean of a transformed variable is equal; the observed association is compatible with random label assignment.

A small p-value can coexist with an astrophysically tiny effect in a huge sample. Conversely, an important effect can fail to achieve a conventional threshold in a small sample. Always report effect size and uncertainty alongside significance.

> **Discussion checkpoint.** Identify the estimand, observed data, and strongest modelling assumption in the example just discussed.

### Worked computational demonstration — Permutation test

Run the cell, then change one assumption and explain what changes statistically.

In [ ]:
import numpy as np
from scipy import stats
import sys
sys.path.append('../src')
from course_utils import load_scalar_sample
df=load_scalar_sample()

x=df['inputs__mass_ratio'].to_numpy()
g=df['inputs__Doppler_boosting'].astype(bool).to_numpy()
obs=x[g].mean()-x[~g].mean()
rng=np.random.default_rng(2026)
null=[]
for _ in range(3000):
    gp=rng.permutation(g)
    null.append(x[gp].mean()-x[~gp].mean())
null=np.asarray(null)
p=(1+np.sum(np.abs(null)>=abs(obs)))/(len(null)+1)
print('effect=',obs,'permutation p=',p)

## 2. Parametric, distribution-free and randomisation tests


A t-test asks about a mean under specific assumptions. A Kolmogorov-Smirnov statistic compares empirical cumulative distributions through the maximum vertical separation,

$$
D_{n,m}=\sup_x |F_n(x)-G_m(x)|.
$$

The Anderson-Darling family places more weight in the tails. None of these tests magically identifies *why* two distributions differ.

Permutation tests are especially transparent. Under exchangeability of labels under $H_0$, repeatedly shuffle the group labels, recompute the statistic, and compare the observed statistic with this empirical null distribution. The method shifts the burden from a distributional formula to the assumption of exchangeability.

This is useful in a simulation course because we can deliberately create cases where the analytic assumptions fail and see how robust the randomisation approach remains.

### Worked computational demonstration — Bootstrap confidence interval

Run the cell, then change one assumption and explain what changes statistically.

In [ ]:
rng=np.random.default_rng(2026)
vals=df['inputs__redshift'].to_numpy()
boot=np.array([np.median(rng.choice(vals,size=len(vals),replace=True)) for _ in range(3000)])
print(np.quantile(boot,[0.025,0.5,0.975]))

## 3. Bootstrap as an uncertainty engine


The nonparametric bootstrap approximates repeated sampling by drawing $n$ observations with replacement from the empirical sample. If $\hat\theta=T(D)$, then

$$
\hat\theta^{\ast b}=T(D^{\ast b})
$$

forms an empirical sampling distribution.

Bootstrap intervals are powerful but not automatic. If data are correlated, rows are not the correct resampling unit. For time series one may need block bootstrap methods; for simulations generated in families, the family may be the independent unit.

This connects to a recurring course rule: **identify the independent unit before splitting, resampling or cross-validating**.

### Worked computational demonstration — Benjamini-Hochberg helper

Run the cell, then change one assumption and explain what changes statistically.

In [ ]:
def benjamini_hochberg(pvals, alpha=0.05):
    p=np.asarray(pvals,float)
    order=np.argsort(p)
    ranked=p[order]
    thresh=alpha*np.arange(1,len(p)+1)/len(p)
    ok=ranked<=thresh
    rejected=np.zeros(len(p),dtype=bool)
    if np.any(ok):
        k=np.max(np.where(ok)[0])
        rejected[order[:k+1]]=True
    return rejected

example=[.001,.007,.02,.04,.20,.80]
print(benjamini_hochberg(example))

## 4. Multiple testing and the look-elsewhere problem


If $m$ independent true null hypotheses are tested at level $\alpha$, the expected number of false positives is $m\alpha$. Astronomy often searches many frequencies, many sources, many features or many sky positions. The nominal p-value of the most interesting result therefore cannot be interpreted as if only one test had been performed.

Family-wise error rate procedures such as Bonferroni are conservative. False-discovery-rate control asks a different question: among reported discoveries, what fraction do we tolerate being false? The Benjamini-Hochberg rule sorts p-values $p_{(1)}\le\cdots\le p_{(m)}$ and finds the largest $k$ satisfying

$$
p_{(k)}\le \frac{k}{m}\alpha.
$$

The same idea will return in period searches, hyperparameter exploration and large survey catalogues.

## 5. SMBH mechanism comparisons


The simulator provides labels for Doppler boosting, lensing flares and accretion modulation. These are useful for controlled experiments, but they should not be confused with labels available from a real survey.

A good Week-2 question is: **does a simulator mechanism induce a measurable population shift in an observable or physical parameter?** Students should pre-specify the variable, statistic and null, estimate an effect size, perform a test, and then repeat the analysis over several variables to expose the multiple-testing problem.

The strongest conclusion is not "p < 0.05". It is a statement that combines the effect magnitude, sampling uncertainty, multiplicity, and the limitations of using simulator-labelled groups.

## Deep dive — errors, power and the design of a scientific test

Classical testing distinguishes a Type-I error (rejecting a true null) from a Type-II error (failing to reject a false null). If the rejection threshold is $\alpha$, statistical power is

$$
1-\beta=P(\text{reject }H_0\mid H_1\text{ is true}).
$$

Power depends on sample size, noise, effect magnitude and the chosen test statistic. This gives a useful perspective on astronomical non-detections: "not significant" does not imply "no effect." It may mean the experiment had insufficient power.

Before collecting or analysing data, one can perform a power simulation: specify a plausible alternative, repeatedly generate data, run the intended test and estimate the rejection rate. In simulations this is easy and should become a habit.

### Confidence intervals and tests are related but not interchangeable

For many regular two-sided tests, rejecting $H_0:\theta=\theta_0$ at level 0.05 corresponds to a 95% confidence interval excluding $\theta_0$. But the confidence interval communicates effect scale and uncertainty much better than a binary reject/fail-to-reject label.

For the SMBH course, if two mechanism groups differ in median mass ratio by only 0.005 but a huge sample makes the p-value tiny, the scientifically relevant statement is that the effect is small. Conversely, a difference of 0.15 with a broad interval may be astrophysically interesting even if a threshold is not crossed.

### Pre-specification versus exploration

Exploratory analysis is legitimate, but a hypothesis generated after seeing the data should not be presented as though it was pre-specified. In a large multidimensional simulation one can always find an unusual subgroup or extreme correlation. The cure is not to forbid exploration; it is to separate exploratory discovery from confirmatory testing and, when possible, validate on an independent subset.

This distinction foreshadows the train/validation/test discipline of machine learning.

## Binary-SMBH synthesis questions

Answer verbally before leaving the lecture.

1. Which quantity in today's topic is a **property of the simulator**, and which could be an **observable property of a real survey**?
2. What uncertainty or selection effect would most strongly distort the result?
3. What diagnostic would you require before trusting the inference?
4. What information would be unavailable if these were real LSST sources rather than simulations?

## Key equations / ideas to retain

Create your own 6–10 line summary here during class.

## Further reading

- Crete `02_Hypothesis_Testing/Hypothesis.ipynb`.
- Benjamini & Hochberg (1995), false discovery rate.
- Feigelson & Babu, chapters on resampling and testing.

# Part II — Extended lecture development

## 11. Choosing the statistic is part of the hypothesis

"The populations are the same" can be probed through differences in means,
medians, variances, tail fractions or an entire-distribution statistic. Different
statistics have different power against different alternatives.

A mean difference is sensitive to location shifts; a variance statistic targets
dispersion; KS detects the largest CDF separation; a maximum statistic targets
extremes. The statistic should correspond to the scientific departure of
interest and should be chosen before its observed extremeness is known.

## 12. One-sided versus two-sided alternatives

If a theory predicts that a mechanism can only increase an observable, a
one-sided alternative can be justified. If the direction is chosen after
seeing the sign of the effect, the nominal one-sided p-value is too optimistic.

For a symmetric null statistic,

$$
p_{\rm two}
=
P(|T^\ast|\ge |T_{\rm obs}|\mid H_0).
$$

The tail choice follows the hypothesis, not the data.

## 13. Confidence intervals as repeated-sampling procedures

A 95% confidence interval is produced by a procedure that covers the true
parameter in 95% of repeated comparable experiments. It does not assign 95%
probability to the fixed parameter after one dataset has been observed.

This differs from a Bayesian credible interval. Both are useful, but their
probability statements should not be mixed.

Intervals show which effect sizes remain plausible under the procedure and are
usually more informative than a binary reject/fail-to-reject label.

## 14. Parametric versus non-parametric bootstrap

A non-parametric bootstrap resamples observed independent units. A parametric
bootstrap fits a model, simulates from it, and repeats the estimator/test.

For fitted-distribution tests, a parametric bootstrap should generally **refit
the model in every simulated sample**, because the observed discrepancy was
calculated after parameter fitting.

Resampling never removes the need to identify the correct independent unit.

In [ ]:
# Parametric-bootstrap calibration of a KS-like discrepancy
rng=np.random.default_rng(2026)
x=np.log10(df["inputs__total_mass"].astype(float).to_numpy())
mu_hat=x.mean(); sd_hat=x.std(ddof=0)
obs_D=stats.kstest(x,"norm",args=(mu_hat,sd_hat)).statistic

B=400
Dsim=np.empty(B)
for b in range(B):
    xb=rng.normal(mu_hat,sd_hat,len(x))
    m=xb.mean(); s=xb.std(ddof=0)
    Dsim[b]=stats.kstest(xb,"norm",args=(m,s)).statistic

print("bootstrap-calibrated p:",
      (1+np.sum(Dsim>=obs_D))/(B+1))

## 15. Dependence invalidates naive standard errors

Time-series points, repeated observations of the same source and grouped
simulations are not independent rows. Positive dependence reduces effective
information, so treating every row as independent typically produces
overconfident uncertainty.

Solutions depend on structure: block bootstrap, cluster resampling, hierarchical
models or an explicit covariance likelihood.

The row count is not automatically the statistical sample size.

## 16. Testing after exploration

Trying many transformations, subgroups and features and reporting the smallest
p-value ignores the selection process. This is another look-elsewhere effect.

Two defensible approaches are: validate the chosen hypothesis on independent
data, or reproduce the entire exploratory selection procedure under the null.

This same logic later applies to hyperparameter tuning and period searches.

## 17. FWER versus FDR

Family-wise error rate controls the probability of at least one false rejection
in a family; Bonferroni is a simple method. False discovery rate controls the
expected fraction of false discoveries among reported discoveries.

Large surveys may prefer FDR because many true signals can exist. A follow-up
catalogue requiring high purity may prefer stricter control. The error criterion
should follow the scientific use of the discoveries.

## 18. Power curves and experimental design

Power is

$$
P(\text{reject }H_0\mid H_1).
$$

Instead of only analysing a realised p-value, simulate power over scientifically
relevant effect sizes. This reveals what the experiment could actually detect.

A non-significant test with 20% power for the effect of interest cannot support
a strong "no effect" conclusion.

In [ ]:
rng=np.random.default_rng(4)
def power(delta,n=60,B=1000):
    hit=0
    for _ in range(B):
        a=rng.normal(0,1,n)
        b=rng.normal(delta,1,n)
        hit += stats.ttest_ind(a,b,equal_var=False).pvalue < .05
    return hit/B

deltas=np.linspace(0,1,11)
plt.plot(deltas,[power(d) for d in deltas],marker="o")
plt.axhline(.8,ls="--")
plt.xlabel("true standardized mean shift"); plt.ylabel("power")
plt.show()

## 19. Reporting template

A complete Week-2 result contains: pre-specified null/alternative, statistic,
group sizes, effect estimate, uncertainty interval, p-value calibration method,
multiplicity correction, physical interpretation and limitations.

Avoid "the null is true" after a non-significant result. Say instead that the
data do not provide strong evidence against the specified null and report which
effect sizes remain compatible with the uncertainty.

# Part III — Worked board problems and discussion

## Problem A — p-value versus posterior probability

A test of $H_0$ returns $p=0.01$.

**Question:** Which of the following is justified?

1. There is a 1% probability that $H_0$ is true.
2. If $H_0$ generated repeated comparable datasets, a statistic at least this
   extreme would occur about 1% of the time.
3. There is a 99% probability that $H_1$ is true.

Only statement 2 follows from the p-value definition. Statements 1 and 3
require prior model probabilities and a Bayesian model comparison.

Ask students to explain why this distinction becomes especially important for
rare phenomena. Even an apparently "significant" signature can have a modest
posterior probability if the alternative is extremely rare or if many targets
were searched.

## Problem B — multiple period searches

Suppose 10,000 independent null sources are each tested once at
$\alpha=0.01$. The expected number of false positives is

$$
10^4\times0.01=100.
$$

Now suppose each source is searched at many frequencies and the smallest
single-frequency p-value is reported without global calibration. The false
candidate count can be much larger.

This is why survey discovery pipelines should distinguish:
- local significance;
- per-source global significance;
- catalogue-level false-discovery control.

Have students propose which level matters for a published candidate catalogue
and which level matters for selecting 20 sources for follow-up.

## Problem C — bootstrap the correct unit

You have 500 sources with 50 observations each. To estimate uncertainty in the
mean source variability amplitude, should you resample the 25,000 observations
or the 500 sources?

If the target is a population mean over sources, the independent population
units are sources. Resampling individual epochs treats correlated observations
as independent and usually underestimates uncertainty.

This problem should be solved before students write any bootstrap code.